# 🎬 Produção Automática de Vídeo para YouTube
## Google Colab - Processamento em Nuvem GRÁTIS

Este notebook recebe os arquivos gerados pelo agente Python e produz o vídeo final.

**Fluxo:**
1. Upload dos arquivos (áudio + imagens + legendas)
2. Processamento automático
3. Download do vídeo final + thumbnail

## 📦 CÉLULA 1: Instalar Dependências

In [ ]:
!pip install moviepy pydub pillow requests -q
!apt-get update -qq && apt-get install ffmpeg imagemagick -y -qq
print("✅ Dependências instaladas!")

## ⚙️ CÉLULA 2: Importar Bibliotecas

In [ ]:
from google.colab import files
from IPython.display import display, Audio, Video
import os, json, random
from PIL import Image, ImageDraw, ImageFont
from moviepy.editor import *

# Configurar pasta de trabalho
WORK_DIR = "/content/youtube_production"
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print(f"✅ Ambiente pronto em: {WORK_DIR}")

## 📤 CÉLULA 3: Upload dos Arquivos

**Envie:**
- Áudio (MP3 gerado pelo agente)
- Imagens (JPG/PNG)
- Legendas (SRT opcional)

In [ ]:
print("📤 Faça upload dos arquivos gerados pelo agente Python")
print("   - audio_*.mp3")
print("   - stock_*.jpg")
print("   - legendas_*.srt")
print()\nuploaded = files.upload()

# Listar arquivos
print("\n📁 Arquivos recebidos:")
for f in uploaded.keys():
    print(f"  • {f}")

## 🎵 CÉLULA 4: Carregar Áudio e Configurar

In [ ]:
# Encontrar arquivo de áudio
audio_file = None
for f in uploaded.keys():
    if f.endswith('.mp3') or f.endswith('.wav'):
        audio_file = f
        break

if not audio_file:
    raise ValueError("Nenhum arquivo de áudio encontrado!")

# Carregar áudio
audio = AudioFileClip(audio_file)
duracao = audio.duration

print(f"✅ Áudio carregado: {audio_file}")
print(f"⏱️ Duração: {duracao:.1f}s ({duracao/60:.1f} min)")

# Preview
display(Audio(audio_file))

## 🖼️ CÉLULA 5: Carregar Imagens

In [ ]:
# Listar imagens
imagens = []
for f in uploaded.keys():
    if f.endswith('.jpg') or f.endswith('.png'):
        imagens.append(f)

if not imagens:
    print("⚠️ Nenhuma imagem encontrada. Criando placeholders...")
    # Criar imagens de placeholder
    for i in range(5):
        img = Image.new('RGB', (1920, 1080), 
                       color=(random.randint(50,150), random.randint(50,150), random.randint(50,150)))
        img.save(f'placeholder_{i}.jpg')
        imagens.append(f'placeholder_{i}.jpg')

print(f"\n📸 Imagens encontradas: {len(imagens)}")
for img in sorted(imagens):
    print(f"  • {img}")

## 🎬 CÉLULA 6: Montar Vídeo

In [ ]:
print("🎬 Montando vídeo...")

# Calcular duração por cena
num_cenas = len(imagens)
duracao_cena = duracao / num_cenas

clips = []
for img in imagens:
    # Criar clip da imagem
    clip = ImageClip(img).set_duration(duracao_cena)
    
    # Efeito Ken Burns (zoom suave)
    clip = clip.resize(lambda t: 1 + 0.03*t)
    
    clips.append(clip)

# Concatenar cenas
video = concatenate_videoclips(clips, method="compose")
video = video.set_duration(duracao)

# Adicionar áudio
video = video.set_audio(audio)

# Transições fade in/out
video = video.fadein(1).fadeout(2)

print(f"✅ Montagem concluída ({num_cenas} cenas)")

## 🏷️ CÉLULA 7: Adicionar Legendas (Opcional)

In [ ]:
# Verificar se tem legendas
legenda_file = None
for f in uploaded.keys():
    if f.endswith('.srt'):
        legenda_file = f
        break

if legenda_file:
    print(f"📝 Adicionando legendas: {legenda_file}")
    
    # Função para gerar texto das legendas
    def make_txtclip(txt, size, color='white', font='Arial', fontsize=40):
        txtclip = TextClip(txt, fontsize=fontsize, color=color, font=font)
        txtclip = txtclip.set_position(('center', 'bottom'))
        return txtclip
    
    # Criar SubtitlesClip
    subs = SubtitlesClip(legenda_file, make_txtclip)
    subs = subs.set_position(('center', 'bottom'))
    
    # Combinar com vídeo
    video = CompositeVideoClip([video, subs])
    print("✅ Legendas adicionadas!")
else:
    print("ℹ️ Nenhuma legenda encontrada. Pulando etapa.")

## 🎞️ CÉLULA 8: Renderizar Vídeo Final

In [ ]:
import time

output_file = "video_final.mp4"

print(f"⏳ Renderizando vídeo... (3-5 minutos)")
start_time = time.time()

video.write_videofile(
    output_file,
    fps=24,
    codec='libx264',
    audio_codec='aac',
    temp_audiofile='temp.m4a',
    remove_temp=True,
    preset='medium',
    bitrate='5000k',
    threads=4
)

elapsed = time.time() - start_time
tamanho_mb = os.path.getsize(output_file) / 1024 / 1024

print(f"\n✅ Vídeo renderizado!")
print(f"   Arquivo: {output_file}")
print(f"   Tamanho: {tamanho_mb:.1f} MB")
print(f"   Tempo: {elapsed/60:.1f} minutos")

## 🖼️ CÉLULA 9: Gerar Thumbnail

In [ ]:
from PIL import Image, ImageDraw, ImageFont

titulo = input("Título do vídeo: ") or "Meu Vídeo YouTube"
nicho = input("Tema (ex: educacao, tecnologia): ") or "educacao"

# Criar thumbnail 1280x720
width, height = 1280, 720
img = Image.new('RGB', (width, height), color=(25, 25, 110))
draw = ImageDraw.Draw(img)

# Gradiente de fundo
for y in range(height):
    r = int(25 + (y/height) * 60)
    g = int(25 + (y/height) * 60)
    b = int(110 + (y/height) * 100)
    draw.line([(0, y), (width, y)], fill=(r, g, b))

# Texto grande
try:
    font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 70)
except:
    font = ImageFont.load_default()

text = titulo.upper()[:40]
bbox = draw.textbbox((0, 0), text, font=font)
x = (width - (bbox[2] - bbox[0])) // 2
y = (height - (bbox[3] - bbox[1])) // 2

# Sombra e texto
draw.text((x+3, y+3), text, fill='black', font=font)
draw.text((x, y), text, fill='white', font=font)

thumbnail_file = "thumbnail.png"
img.save(thumbnail_file, 'PNG')

print(f"✅ Thumbnail gerada: {thumbnail_file}")
display(Image.open(thumbnail_file))

## 📥 CÉLULA 10: Preview e Download

In [ ]:
print("🎬 Preview do vídeo final:\n")
display(Video(output_file, width=640))

print("\n📥 Baixando arquivos...")
files.download(output_file)
files.download(thumbnail_file)

print("\n" + "="*50)
print("  ✅ PRODUÇÃO CONCLUÍDA!")
print("="*50)
print(f"\n📁 Arquivos gerados:")
print(f"   • Vídeo: {output_file} ({tamanho_mb:.1f} MB)")
print(f"   • Thumbnail: {thumbnail_file}")
print(f"\n🚀 Próximo passo: Upload no YouTube Studio!")